Esqueleto

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

%matplotlib inline

In [ ]:
class Coupled_Oscillators:
    def __init__(self, N, m=1.0, k_springs=1.0, left_wall_k=0.0, right_wall_k=0.0):
        self.N = N
        self.M = self._build_mass_matrix(m)
        self.K = self._build_K_matrix(k_springs, left_wall_k, right_wall_k)
        # a matriz de massa e diagonal, por isso a inversa e so 1/diagonal
        self.Minv = np.diag(1.0 / np.diag(self.M))

    def _build_mass_matrix(self, m):
        # m pode ser um unico valor (massas iguais) ou um array com as N massas
        if np.isscalar(m):
            if m <= 0:
                raise ValueError("a massa tem de ser positiva")
            return np.diag(np.full(self.N, float(m)))
        m = np.asarray(m, dtype=float)
        if len(m) != self.N:
            raise ValueError(f"m deve ter {self.N} elementos")
        return np.diag(m)

    def _build_K_matrix(self, k_springs, left_wall_k, right_wall_k):
        # k_springs pode ser um valor (molas iguais) ou um array com as N-1 molas
        if np.isscalar(k_springs):
            k = np.full(self.N - 1, float(k_springs))
        else:
            k = np.asarray(k_springs, dtype=float)
            if len(k) != self.N - 1:
                raise ValueError(f"k_springs deve ter {self.N - 1} elementos")

        kL, kR = left_wall_k, right_wall_k
        if np.any(k <= 0) or kL < 0 or kR < 0:
            raise ValueError("as constantes das molas tem de ser positivas")

        # diagonal principal: soma das molas ligadas a cada massa (com paredes nos extremos)
        diag = np.empty(self.N)
        diag[0] = kL + k[0]
        diag[-1] = k[-1] + kR
        for i in range(1, self.N - 1):
            diag[i] = k[i - 1] + k[i]

        K = np.zeros((self.N, self.N))
        np.fill_diagonal(K, diag)
        # molas entre massas vizinhas (diagonais acima e abaixo)
        for i in range(self.N - 1):
            K[i, i + 1] = -k[i]
            K[i + 1, i] = -k[i]
        return K

    def _matrix_A(self):
        # passa o sistema de 2a ordem para 1a ordem: u' = A u, com u = [x, v]
        N = self.N
        zero = np.zeros((N, N))
        return np.block([[zero,                np.eye(N)],
                         [-self.Minv @ self.K, zero]])

    def solve_coupled_system_linear(self, x0=None, v0=None, t_max=50, num_points=1000):
        if x0 is None:
            x0 = np.zeros(self.N)
        if v0 is None:
            v0 = np.zeros(self.N)
        x0 = np.asarray(x0, dtype=float)
        v0 = np.asarray(v0, dtype=float)
        if len(x0) != self.N:
            raise ValueError(f"x0 tem de ter {self.N} elementos")

        A = self._matrix_A()
        u0 = np.concatenate([x0, v0])
        t = np.linspace(0, t_max, num_points)

        # metodo do scipy (RK45)
        sol = solve_ivp(lambda t, u: A @ u, (0, t_max), u0, t_eval=t, method='RK45')
        if not sol.success:
            raise RuntimeError(sol.message)
        sol_t, sol_u = sol.t, sol.y

        # metodo de Euler
        dt = t_max / (num_points - 1)
        sol_euler_u = np.zeros((2 * self.N, num_points))
        sol_euler_u[:, 0] = u0
        for i in range(num_points - 1):
            sol_euler_u[:, i + 1] = sol_euler_u[:, i] + dt * (A @ sol_euler_u[:, i])

        return sol_t, sol_u, t, sol_euler_u

    def solve_rk45(self, x0, v0, t_max=50, num_points=1000):
        # so o RK45, para quando nao precisamos da solucao de Euler
        A = self._matrix_A()
        u0 = np.concatenate([np.asarray(x0, dtype=float), np.asarray(v0, dtype=float)])
        t = np.linspace(0, t_max, num_points)
        sol = solve_ivp(lambda t, u: A @ u, (0, t_max), u0, t_eval=t, method='RK45')
        return sol.t, sol.y

    def normal_modes(self):
        # problema generalizado K eta = lambda M eta, com lambda = omega^2
        # como M e diagonal, passamos a um problema simetrico standard C y = lambda y,
        # com C = M^(-1/2) K M^(-1/2) e eta = M^(-1/2) y -> usamos numpy.linalg.eigh
        m = np.diag(self.M)
        M_half_inv = np.diag(1.0 / np.sqrt(m))
        C = M_half_inv @ self.K @ M_half_inv
        lambdas, ys = np.linalg.eigh(C)   # ja vem ordenado e com vetores ortonormais
        etas = M_half_inv @ ys            # modos normalizados a massa: eta^T M eta = 1
        omegas = np.sqrt(np.abs(lambdas))
        return omegas, etas

Método de Euler

Vamos testar os nossos métodos numa EDO simples com solução analítica conhecida:

$$\frac{dy}{dx} = xy, \quad y(1) = 0$$

A solução analítica é $y(t) = e^{\frac{x^2}{2}}$.

In [ ]:
if "PALETTE" not in dir(): PALETTE = ["#e63946", "#457b9d", "#2a9d8f", "#e9c46a", "#f4a261"]
def metodo_euler(f, t0, y0, t_final, h):
    """
    Resolver a EDO pelo método de Euler, para comparação
    f - a EDO que queremos resolver
    t0 - abcissa do ponto inicial
    y0 - ordenada do ponto inicial
    t_final - ordenada do ponto final
    h - passo, precisão de resolução
    """
    X = np.arange(t0, t_final + h, h)
    # Criação de um eixo das abcissas dividido por steps

    Y = np.zeros(len(X))
    # "Cópia" estrutural de do eixo criado

    Y[0] = y0
    # Ponto inicial a partir do qual vamos avaliar o método
    # Ou seja, precisamos de pelo menos um ponto que conhecemos ser uma solução
    
    for i in range(len(X) - 1):
        Y[i+1] = Y[i] + h * f(X[i], Y[i]) 
        # Fazer o contrário de uma derivada
        # Só funciona para EDO's

    return X, Y

def eq_dif(x, y):
    return x * y

def sol_analítica(x, y):
    return np.exp(x**2 / 2)

X = np.arange(0, 1 + 0.02, 0.02)
Y = sol_analítica(X, 0)

plt.plot(X, Y, label="analítica", c="black")

x_euler, y_euler = metodo_euler(eq_dif, 0, 1, 1, 0.02)
plt.plot(x_euler, y_euler, 'ro-', label='euler')
plt.legend()
plt.grid(True, ls="--")
plt.show()


Assumindo todas as massas iguais (escolhe um valor apropriado), todas as molas iguais
(escolhe um valor apropriado) e incluindo molas ligadas a paredes de ambos os lados, simule um
sistema com 5 massas. Como condições iniciais assume que todas as massas se encontram em
repouso e que apenas a massa no centro (m3) se encontra fora da posição de equilíbrio (por um
deslocamento que julgues adequado). Representa graficamente a posição de cada uma das massas
ao longo do tempo (considerar um tempo final que julgues apropriado). Compara as soluções
obtidas pelo método de Euler e pelo método implementado no scipy.integrate

Dica para o comentario: as duas solucoes coincidem no inicio e o Euler vai-se afastando do RK45 a medida que o tempo passa (o erro acumula).

In [ ]:
if "PALETTE" not in dir(): PALETTE = ["#e63946", "#457b9d", "#2a9d8f", "#e9c46a", "#f4a261"]
N = 5
m_global = 1
k_global, k_right, k_left = 1, 1, 1
num_points = 3000
t_max = 30
initial_x = [0 for _ in range(N)]
initial_v = np.zeros_like(initial_x)
initial_x[2] = 1

sistema = Coupled_Oscillators(N, m_global, k_global, k_left, k_right)
sol_t_np, sol_u_np, sol_t_euler, sol_u_euler = sistema.solve_coupled_system_linear(initial_x, initial_v, t_max, num_points)

comparison = sol_u_np - sol_u_euler # Onde é que diferem e a partir de quando

# Extrair posições e velocidades
x_np    = sol_u_np[:N, :]
v_np    = sol_u_np[N:, :]
x_euler = sol_u_euler[:N, :]
v_euler = sol_u_euler[N:, :]

colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#f4a261']
labels = [f'm{i+1}' for i in range(N)]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='#0f0f0f')
fig.suptitle('Coupled Oscillators', color='white', fontsize=18, fontweight='bold', y=0.98)

titles = [('NumPy (RK45) — Posições', 'Posição (m)'),
          ('NumPy (RK45) — Velocidades', 'Velocidade (m/s)'),
          ('Euler — Posições', 'Posição (m)'),
          ('Euler — Velocidades', 'Velocidade (m/s)')]

data = [(sol_t_np, x_np), (sol_t_np, v_np),
        (sol_t_euler, x_euler), (sol_t_euler, v_euler)]

for ax, (t, y), (title, ylabel) in zip(axes.flat, data, titles):
    ax.set_facecolor('#1a1a1a')
    for i in range(N):
        ax.plot(t, y[i, :], color=colors[i], label=labels[i], linewidth=0.8, alpha=0.9)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel('Tempo (s)', color='#aaaaaa', fontsize=9)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333333')
    ax.legend(fontsize=8, facecolor='#111111', labelcolor='white', framealpha=0.7)
    ax.grid(True, color='#2a2a2a', linewidth=0.5)


x_diff = comparison[:N, :]
v_diff = comparison[N:, :]

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0f0f0f')
fig2.suptitle('Diferença RK45 − Euler', color='white', fontsize=18, fontweight='bold', y=1.02)

data_diff  = [(sol_t_np, x_diff, 'Posições', 'Diferença (m)'),
              (sol_t_np, v_diff, 'Velocidades', 'Diferença (m/s)')]

for ax, (t, y, title, ylabel) in zip(axes2.flat, data_diff):
    ax.set_facecolor('#1a1a1a')
    for i in range(N):
        ax.plot(t, y[i, :], color=colors[i], label=labels[i], linewidth=0.8, alpha=0.9)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel('Tempo (s)', color='#aaaaaa', fontsize=9)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333333')
    ax.legend(fontsize=8, facecolor='#111111', labelcolor='white', framealpha=0.7)
    ax.grid(True, color='#2a2a2a', linewidth=0.5)


plt.tight_layout()
plt.show()

Repete a simulação com condições iniciais em que as massas estão todas em repouso, mas
em que estão todas deslocadas pela mesma distância para um lado. Existe alguma diferença com os resultados da alínea anterior? Comenta

Dica para o comentario: com paredes dos dois lados nao existe modo de translacao livre, por isso o sistema oscila a mesma em vez de deslizar em bloco (compara com a alinea anterior).

In [ ]:
if "PALETTE" not in dir(): PALETTE = ["#e63946", "#457b9d", "#2a9d8f", "#e9c46a", "#f4a261"]
N = 5
m_global = 1
k_global, k_right, k_left = 1, 1, 1
num_points = 3000
t_max = 30
initial_x = [1 for _ in range(N)]
initial_v = np.zeros_like(initial_x)

sistema = Coupled_Oscillators(N, m_global, k_global, k_left, k_right)
sol_t_np, sol_u_np, sol_t_euler, sol_u_euler = sistema.solve_coupled_system_linear(initial_x, initial_v, t_max, num_points)

comparison = sol_u_np - sol_u_euler # Onde é que diferem e a partir de quando

# Extrair posições e velocidades
x_np    = sol_u_np[:N, :]
v_np    = sol_u_np[N:, :]
x_euler = sol_u_euler[:N, :]
v_euler = sol_u_euler[N:, :]



colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#f4a261']
labels = [f'm{i+1}' for i in range(N)]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='#0f0f0f')
fig.suptitle('Coupled Oscillators', color='white', fontsize=18, fontweight='bold', y=0.98)

titles = [('NumPy (RK45) — Posições', 'Posição (m)'),
          ('NumPy (RK45) — Velocidades', 'Velocidade (m/s)'),
          ('Euler — Posições', 'Posição (m)'),
          ('Euler — Velocidades', 'Velocidade (m/s)')]

data = [(sol_t_np, x_np), (sol_t_np, v_np),
        (sol_t_euler, x_euler), (sol_t_euler, v_euler)]

for ax, (t, y), (title, ylabel) in zip(axes.flat, data, titles):
    ax.set_facecolor('#1a1a1a')
    for i in range(N):
        ax.plot(t, y[i, :], color=colors[i], label=labels[i], linewidth=0.8, alpha=0.9)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel('Tempo (s)', color='#aaaaaa', fontsize=9)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333333')
    ax.legend(fontsize=8, facecolor='#111111', labelcolor='white', framealpha=0.7)
    ax.grid(True, color='#2a2a2a', linewidth=0.5)


x_diff = comparison[:N, :]
v_diff = comparison[N:, :]

fig2, axes2 = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0f0f0f')
fig2.suptitle('Diferença RK45 − Euler', color='white', fontsize=18, fontweight='bold', y=1.02)

data_diff  = [(sol_t_np, x_diff, 'Posições', 'Diferença (m)'),
              (sol_t_np, v_diff, 'Velocidades', 'Diferença (m/s)')]

for ax, (t, y, title, ylabel) in zip(axes2.flat, data_diff):
    ax.set_facecolor('#1a1a1a')
    for i in range(N):
        ax.plot(t, y[i, :], color=colors[i], label=labels[i], linewidth=0.8, alpha=0.9)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel('Tempo (s)', color='#aaaaaa', fontsize=9)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333333')
    ax.legend(fontsize=8, facecolor='#111111', labelcolor='white', framealpha=0.7)
    ax.grid(True, color='#2a2a2a', linewidth=0.5)


plt.tight_layout()
plt.show()

A energia cinética é dada por:
$$T(t) = \frac{1}{2}\sum^{N}_{i=1}m_i \dot{x}_i^2(t) = \frac{1}{2}\bold{v}^T(t)\bold{M}\bold{v}(t)$$
A energia potencial é dada por:
$$V(t) = \frac{1}{2}k_L\dot{x}_1^2(t)+\frac{1}{2}\sum_{i=1}^{N-1}k_{i,i+1}(x_{i+1}(t)-x_i(t))^2 + \frac{1}{2}k_R x_N^2(t) = \frac{1}{2}\bold{x}^T(t)\bold{Kx}(t)$$

Represente graficamente a evoluçaõ no tempo da energia cinética $T$, da energia potencial $V$, e da energia mecânica total $E = T + V$. Sabendo que a energia total é conservada, compare a precisão dos dois métodos usados para reslver as equaç~çoes diferenciais. 
Em princípio o método que conserva melhor a energia mecância total será o que tem mais precisão.

Vamos aplicá-lo à simulação corrida anteriormente.

Dica para o comentario: ver qual metodo mantem E = T + V mais constante. O que conserva melhor a energia e o mais preciso (espera-se Euler a divergir e RK45 quase constante).

In [ ]:
if "PALETTE" not in dir(): PALETTE = ["#e63946", "#457b9d", "#2a9d8f", "#e9c46a", "#f4a261"]
def kinetic(vel, M):
    # T = 1/2 v^T M v calculado para todos os instantes de uma vez (vel tem shape (N, num_points))
    return 0.5 * np.sum(vel * (M @ vel), axis=0)

def potencial(pos, K):
    # V = 1/2 x^T K x para todos os instantes
    return 0.5 * np.sum(pos * (K @ pos), axis=0)

np_potencial = potencial(x_np, sistema.K)
np_kinetic = kinetic(v_np, sistema.M)
np_mec = np_potencial + np_kinetic

euler_potencial = potencial(x_euler, sistema.K)
euler_kinetic = kinetic(v_euler, sistema.M)
euler_mec = euler_potencial + euler_kinetic

fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5), facecolor='#0f0f0f')
fig3.suptitle('Energia — RK45 vs Euler', color='white', fontsize=18, fontweight='bold')

energy_data = [
    (np_kinetic,   euler_kinetic,   'Energia Cinética',   'E cinética (J)'),
    (np_potencial, euler_potencial, 'Energia Potencial',  'E potencial (J)'),
    (np_mec,       euler_mec,       'Energia Mecânica',   'E mecânica (J)'),
]

for ax, (e_np, e_euler, title, ylabel) in zip(axes3, energy_data):
    ax.set_facecolor('#1a1a1a')
    ax.plot(sol_t_np, e_np,    color='#457b9d', label='RK45',  linewidth=1.0)
    ax.plot(sol_t_np, e_euler, color='#e63946', label='Euler', linewidth=1.0, alpha=0.8)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel('Tempo (s)', color='#aaaaaa', fontsize=9)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333333')
    ax.legend(fontsize=9, facecolor='#111111', labelcolor='white', framealpha=0.7)
    ax.grid(True, color='#2a2a2a', linewidth=0.5)

plt.tight_layout()
plt.show()



Modos normais

Os modos normais sao as solucoes em que todas as massas oscilam com a mesma frequencia. Como as solucoes sao harmonicas, o sistema da Eq. 3 transforma-se num problema de valores proprios:

$$M^{-1}K\,\eta_j = \lambda_j\,\eta_j, \qquad \lambda_j = \omega_j^2$$

Os vetores proprios $\eta_j$ sao os modos normais e $\omega_j = \sqrt{\lambda_j}$ as frequencias proprias. Acrescentamos o metodo `normal_modes` a classe, que usa o `numpy.linalg`. Como $M$ e diagonal, reescrevemos o problema na forma simetrica $C\,y = \lambda\,y$ com $C = M^{-1/2} K M^{-1/2}$, resolvido com `numpy.linalg.eigh` (frequencias ordenadas e modos com $\eta_j^T M \eta_j = 1$).

Para cada modo, o movimento das massas e $x_i(t) = \eta_{ij}\cos(\omega_j t)$. Representamos cada modo num plot separado, com a trajetoria de todas as massas.

In [ ]:
if "PALETTE" not in dir(): PALETTE = ["#e63946", "#457b9d", "#2a9d8f", "#e9c46a", "#f4a261"]
# Parte 3a - modos normais
N = 5
m_global = 1
k_global, k_right, k_left = 1, 1, 1

sistema = Coupled_Oscillators(N, m_global, k_global, k_left, k_right)
omegas, modos = sistema.normal_modes()

print("Frequencias proprias:")
for j in range(N):
    print(f"  modo {j+1}: omega = {omegas[j]:.4f}")

# movimento das massas em cada modo: x_i(t) = eta_ij * cos(omega_j * t)
t = np.linspace(0, 30, 1000)

colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#f4a261']
labels = [f'm{i+1}' for i in range(N)]

fig, axes = plt.subplots(N, 1, figsize=(12, 15), facecolor='#0f0f0f')
fig.suptitle('Modos Normais', color='white', fontsize=18, fontweight='bold')

for j in range(N):
    ax = axes[j]
    ax.set_facecolor('#1a1a1a')
    for i in range(N):
        xi = modos[i, j] * np.cos(omegas[j] * t)
        ax.plot(t, xi, color=colors[i], label=labels[i], linewidth=0.9)
    ax.set_title(f'Modo {j+1}  (omega = {omegas[j]:.3f})', color='white', fontsize=11)
    ax.set_xlabel('Tempo (s)', color='#aaaaaa', fontsize=9)
    ax.set_ylabel('Posicao (m)', color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333333')
    ax.legend(fontsize=8, facecolor='#111111', labelcolor='white', framealpha=0.7)
    ax.grid(True, color='#2a2a2a', linewidth=0.5)

plt.tight_layout()
plt.show()

Sobreposicao de modos normais

Qualquer solucao do sistema pode ser escrita como uma sobreposicao linear dos modos normais:

$$x(t) = \sum_{j=1}^{N}\left[\left(\eta_j^T M x_0\right)\cos(\omega_j t) + \left(\frac{\eta_j^T M v_0}{\omega_j}\right)\sin(\omega_j t)\right]\eta_j$$

Usamos as mesmas condicoes iniciais da parte 2a (todas as massas em repouso e so a massa do meio deslocada) e comparamos esta reconstrucao com a solucao numerica obtida na parte 1 (RK45).

Dica para o comentario: as duas curvas devem sobrepor-se quase na perfeicao; a pequena diferenca vem so do erro numerico do RK45.

In [ ]:
if "PALETTE" not in dir(): PALETTE = ["#e63946", "#457b9d", "#2a9d8f", "#e9c46a", "#f4a261"]
# Parte 3b - solucao por sobreposicao dos modos normais
N = 5
m_global = 1
k_global, k_right, k_left = 1, 1, 1
t_max = 30
num_points = 3000

x0 = np.zeros(N)
x0[2] = 1          # so a massa do meio fora do equilibrio
v0 = np.zeros(N)

sistema = Coupled_Oscillators(N, m_global, k_global, k_left, k_right)
omegas, modos = sistema.normal_modes()

# solucao da parte 1 para comparar
sol_t_np, sol_u_np = sistema.solve_rk45(x0, v0, t_max, num_points)
x_np = sol_u_np[:N, :]
t = sol_t_np

# reconstruir x(t) somando a contribuicao de cada modo
x_modos = np.zeros((N, len(t)))
for j in range(N):
    eta = modos[:, j]
    a = eta @ sistema.M @ x0            # coeficiente do cosseno
    b = (eta @ sistema.M @ v0) / omegas[j]   # coeficiente do seno
    temporal = a * np.cos(omegas[j] * t) + b * np.sin(omegas[j] * t)
    x_modos += np.outer(eta, temporal)

colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a', '#f4a261']

fig, axes = plt.subplots(N, 1, figsize=(12, 15), facecolor='#0f0f0f')
fig.suptitle('Sobreposicao de modos normais vs Parte 1 (RK45)', color='white', fontsize=18, fontweight='bold')

for i in range(N):
    ax = axes[i]
    ax.set_facecolor('#1a1a1a')
    ax.plot(t, x_np[i, :], color='#457b9d', label='RK45 (parte 1)', linewidth=1.3)
    ax.plot(t, x_modos[i, :], color='#e63946', label='modos normais', linewidth=0.9, ls='--')
    ax.set_title(f'm{i+1}', color='white', fontsize=11)
    ax.set_xlabel('Tempo (s)', color='#aaaaaa', fontsize=9)
    ax.set_ylabel('Posicao (m)', color='#aaaaaa', fontsize=9)
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333333')
    ax.legend(fontsize=8, facecolor='#111111', labelcolor='white', framealpha=0.7)
    ax.grid(True, color='#2a2a2a', linewidth=0.5)

plt.tight_layout()
plt.show()

# diferenca maxima entre os dois metodos
print(f"Diferenca maxima entre RK45 e sobreposicao de modos: {np.abs(x_np - x_modos).max():.2e}")